# SP-03 Trend-Following Policy — `03_trend_following_policy.ipynb`

**Project:** Personalized AI Trading Agent (research / paper-trading only)
**Artifact:** `trend_following_policy.pt` | **Policy ID:** `SP-03`

## What this notebook does
Trains a GRU/LSTM sequence policy that **enters or maintains long exposure while a trend is supported and reduces or exits it when the trend weakens or reverses**, then evaluates it with a chronological split and a next-bar-execution backtest that includes costs and slippage.

## Read this first (honest scope)
* **Labels are proxy labels** produced by a documented, causal, rule-based trend-following *teacher*, unless you supply real trader actions (`HUMAN_ACTIONS_CSV`). Classification metrics therefore measure **how well the network imitates the rule**, *not* profitability. The backtest measures what the network actually earns.
* Nothing here is investment advice. No result is guaranteed. Past performance, if any shows up below, is not predictive. Use for **research and paper trading** only.
* Results are whatever the run produces. The notebook prints failures (drawdowns, losing years, underperforming tickers) alongside successes.
* Data comes from Yahoo Finance via `yfinance` (adjusted prices, survivorship-biased ticker list chosen by the user, occasional data errors). Failed downloads are reported loudly, never silently dropped.

## How SP-03 differs from the Momentum Policy
| | Momentum policy | **Trend-following (SP-03)** |
|---|---|---|
| Core question | Is price *accelerating* recently? | Is a directional regime *persisting and supported*? |
| Typical inputs | short-horizon returns, ROC changes, RSI/MACD acceleration | MA stack (20/50/200), EMA crossovers, MA slopes, ADX/DI, efficiency ratio, trend persistence & age |
| Position handling | mostly signal-driven | **position-aware**: scale-in, trim, trailing-stop-style exit, cool-down |
| Exits | momentum fade | trend break / trend weakening / trailing ATR stop |

## Action & position conventions (shared with other specialized policies)
`0 = HOLD`, `1 = BUY`, `2 = SELL`, `3 = EXIT`.
Position is **long-only, in discrete exposure units** (default `max_units = 2`, each unit = 50 % of the position limit):
* `BUY` → add one unit (opens a position if flat). Invalid at max units → treated as HOLD.
* `SELL` → **trim one unit** (partial reduction). Invalid when flat → HOLD.
* `EXIT` → close the **entire** position. Invalid when flat → HOLD.

So SELL is *not* an exit: it only reduces exposure; EXIT always goes to zero.

## Timing convention
Features at bar *t* use data up to the **close of t**. The action decided at *t* is executed at the **open of t+1** (next-bar), with commission + slippage.

## 1. Installation

In [ ]:
# Colab already ships torch, pandas, numpy, scikit-learn, matplotlib. yfinance is upgraded because Yahoo changes often.
!pip -q install -U yfinance

## 2. Imports, device, reproducibility

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # needed for deterministic CUDA matmuls

import sys, json, math, time, pickle, random, hashlib, platform, warnings, shutil, copy
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             balanced_accuracy_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight

import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

ENV_INFO = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "yfinance": getattr(yf, "__version__", "unknown"),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV_INFO, indent=2))

## 3. Configuration
Everything configurable lives in `CFG`. It is exported to `trend_following_config.json`.

**Chronological split:** train ≤ `train_end`, validation in (`train_end`, `val_end`], held-out **test > `val_end`**. The test period is never used for fitting, scaling, early stopping or threshold choice.

In [ ]:
CFG = {
    # --- identity ---
    "policy_id": "SP-03",
    "policy_name": "SP-03 Trend-Following Policy",
    "artifact": "trend_following_policy.pt",

    # --- data ---
    "tickers": ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "GLD", "XLE", "AAPL", "MSFT", "JPM", "XOM"],
    "benchmark": "SPY",
    "start": "2004-01-01",
    "end": None,                  # None = latest available
    "min_assets": 4,              # abort if fewer tickers download successfully
    "min_rows_after_warmup": 1500,
    "use_cache": False,           # True: reuse data_cache/*.csv (reproducible, but may be stale)
    "cache_dir": "data_cache",

    # --- chronological split ---
    "train_end": "2016-12-31",
    "val_end": "2020-12-31",      # test = everything after this date

    # --- model ---
    "seq_len": 60,
    "hidden_size": 64,
    "num_layers": 2,
    "rnn_type": "GRU",            # "GRU" or "LSTM"
    "dropout": 0.2,

    # --- training ---
    "seed": 42,
    "batch_size": 256,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 60,
    "patience": 10,               # early stopping on validation loss
    "grad_clip": 1.0,
    "label_smoothing": 0.0,
    "conf_loss_weight": 0.2,      # weight of the confidence-head BCE loss
    "class_weight_mode": "balanced_sqrt",   # "none" | "balanced" | "balanced_sqrt"

    # --- feature windows ---
    "use_trader_features": False, # optional static trader-behaviour / risk-profile features
    "trader_profile": {           # all in [0,1]; only used if use_trader_features=True
        "risk_tolerance": 0.5, "drawdown_tolerance": 0.5,
        "horizon_preference": 0.7, "trade_frequency_preference": 0.3,
    },
    "feature_clip": 5.0,          # clip standardized market features to +/- this

    # --- proxy-label (teacher) rule: past-only ---
    "adx_trend": 25.0,            # ADX needed to call a regime TRENDING
    "adx_add": 30.0,              # ADX needed to scale in
    "recent_trend_window": 40,    # bars during which a lost trend is WEAKENING (else SIDEWAYS)
    "atr_stop_mult": 3.0,         # trailing stop = peak close since entry - mult * ATR
    "weak_exit_bars": 10,         # consecutive WEAKENING bars before last unit is exited
    "cooldown_bars": 5,           # bars to wait after an EXIT before re-entering
    "min_hold_add": 10,           # bars held before scaling in
    "max_units": 2,               # discrete exposure units

    # --- optional real trader actions (CSV with columns: date,ticker,action) ---
    "human_actions_csv": None,    # e.g. "/content/my_trades.csv"; see section 8

    # --- backtest ---
    "initial_capital": 100_000.0,
    "commission_bps": 1.0,        # per side, on notional
    "slippage_bps": 5.0,          # per side, adverse to the trader, applied to the next open
    "max_position_fraction": 1.0, # position limit per asset as a fraction of that asset's equity (no leverage)
    "fractional_shares": True,
    "rf_annual": 0.0,
    "trading_days": 252,
    "run_cost_sensitivity": True,
    "cost_multipliers": [0.0, 1.0, 2.0, 4.0],

    # --- output ---
    "output_dir": "sp03_artifacts",
}

ACTIONS = {0: "HOLD", 1: "BUY", 2: "SELL", 3: "EXIT"}
HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
REGIMES = {0: "SIDEWAYS", 1: "TRENDING", 2: "WEAKENING"}
SIDEWAYS, TRENDING, WEAKENING = 0, 1, 2
POSITION_COLUMNS = ["position_fraction", "unrealized_pnl", "holding_duration", "stop_distance_atr"]

set_seed(CFG["seed"])
OUT = Path(CFG["output_dir"]); OUT.mkdir(exist_ok=True)
TEST_START = pd.Timestamp(CFG["val_end"]) + pd.Timedelta(days=1)
print("Device:", DEVICE, "| test period starts:", TEST_START.date())

## 4. Data download and validation
Each ticker is downloaded separately with retries. Every outcome (success, failure, issues) is recorded in `DOWNLOAD_REPORT` and printed. The notebook **stops** if the benchmark or too many tickers fail — it never substitutes synthetic data.

In [ ]:
def _flatten_cols(df):
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = df.columns.get_level_values(0)
    return df

def clean_ohlcv(df):
    """Validate / clean a raw OHLCV frame. Returns (clean_df, issues_dict)."""
    issues = {}
    idx = pd.DatetimeIndex(df.index)
    if idx.tz is not None:
        idx = idx.tz_convert(None)
    df = df.copy(); df.index = idx.normalize()
    df = df.sort_index()
    dup = int(df.index.duplicated().sum())
    if dup: issues["duplicate_dates_dropped"] = dup; df = df[~df.index.duplicated(keep="last")]
    df = df.replace([np.inf, -np.inf], np.nan)
    n_nan = int(df[["Open", "High", "Low", "Close"]].isna().any(axis=1).sum())
    if n_nan: issues["rows_with_nan_ohlc_dropped"] = n_nan
    df = df.dropna(subset=["Open", "High", "Low", "Close"])
    nonpos = int((df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum())
    if nonpos: issues["nonpositive_price_rows_dropped"] = nonpos
    df = df[(df[["Open", "High", "Low", "Close"]] > 0).all(axis=1)]
    bad = (df["High"] < df[["Open", "Close", "Low"]].max(axis=1) - 1e-9) | (df["Low"] > df[["Open", "Close", "High"]].min(axis=1) + 1e-9)
    if int(bad.sum()): issues["inconsistent_ohlc_rows_dropped"] = int(bad.sum()); df = df[~bad]
    df["Volume"] = df["Volume"].fillna(0.0).clip(lower=0.0)
    if len(df) > 1:
        gap = int(df.index.to_series().diff().dt.days.max())
        if gap > 10: issues["max_calendar_gap_days"] = gap
    return df, issues

def download_ohlcv(ticker, start, end, retries=3):
    last_err = None
    for k in range(retries):
        try:
            raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False, threads=False)
            raw = _flatten_cols(raw)
            if raw is None or raw.empty:
                raise ValueError("empty frame returned")
            need = ["Open", "High", "Low", "Close", "Volume"]
            miss = [c for c in need if c not in raw.columns]
            if miss:
                raise ValueError(f"missing columns {miss}")
            return raw[need], None
        except Exception as e:
            last_err = repr(e)
            time.sleep(2 * (k + 1))
    return None, last_err

def data_fingerprint(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:16]

cache_dir = Path(CFG["cache_dir"]); cache_dir.mkdir(exist_ok=True)
all_symbols = list(dict.fromkeys(CFG["tickers"] + [CFG["benchmark"]]))
RAW, report_rows = {}, []
for sym in all_symbols:
    cache_file = cache_dir / f"{sym.replace('^','_')}.csv"
    raw, err = None, None
    if CFG["use_cache"] and cache_file.exists():
        raw = pd.read_csv(cache_file, index_col=0, parse_dates=True)
    else:
        raw, err = download_ohlcv(sym, CFG["start"], CFG["end"])
    if raw is None:
        report_rows.append(dict(symbol=sym, status="FAILED", rows=0, first=None, last=None, issues=None, error=err))
        continue
    clean, issues = clean_ohlcv(raw)
    clean.to_csv(cache_file)
    if len(clean) < CFG["min_rows_after_warmup"] // 2:
        report_rows.append(dict(symbol=sym, status="TOO_SHORT", rows=len(clean), first=clean.index.min() if len(clean) else None,
                                last=clean.index.max() if len(clean) else None, issues=issues, error="insufficient history"))
        continue
    RAW[sym] = clean
    report_rows.append(dict(symbol=sym, status="OK", rows=len(clean), first=clean.index[0].date(), last=clean.index[-1].date(),
                            issues=issues or None, error=None, fingerprint=data_fingerprint(clean)))

DOWNLOAD_REPORT = pd.DataFrame(report_rows)
print(DOWNLOAD_REPORT.to_string(index=False))
FAILED = DOWNLOAD_REPORT[DOWNLOAD_REPORT.status != "OK"].symbol.tolist()
if FAILED:
    print(f"\n!!! WARNING: {len(FAILED)} symbol(s) not usable: {FAILED}. They are EXCLUDED and will be listed in the exported config/metrics.")
if CFG["benchmark"] not in RAW:
    raise RuntimeError(f"Benchmark {CFG['benchmark']} failed to download; cannot compute benchmark-relative features. Check network / ticker.")
TRAIN_TICKERS = [t for t in CFG["tickers"] if t in RAW]
if len(TRAIN_TICKERS) < CFG["min_assets"]:
    raise RuntimeError(f"Only {len(TRAIN_TICKERS)} usable tickers (< min_assets={CFG['min_assets']}).")
stale_days = (pd.Timestamp.today().normalize() - RAW[CFG["benchmark"]].index[-1]).days
print(f"\nUsable tickers ({len(TRAIN_TICKERS)}): {TRAIN_TICKERS}\nBenchmark last bar is {stale_days} calendar day(s) old.")

## 5. Feature schema and causal feature engineering

All features are computed with **backward-looking** operations only (rolling / EWM / diff / shift with positive lags). A look-ahead unit test below recomputes features on truncated data and checks they are identical.

**Column ordering is fixed by `MARKET_COLUMNS` → `POSITION_COLUMNS` → (optional) `TRADER_COLUMNS`.** This order is exported to `feature_schema.json` and must be reproduced by the Meta-RL selector.

| Group | Features |
|---|---|
| returns | `ret_1, ret_5, ret_21, ret_63, ret_126` (log returns) |
| ma_structure | `px_vs_sma20/50/200`, `sma20_vs_sma50`, `sma50_vs_sma200` |
| ema | `ema12_vs_ema26`, `ema20_vs_ema50` |
| slopes | `sma50_slope_10`, `sma200_slope_20`, `ema20_slope_5` |
| trend_strength | `adx`, `plus_di`, `minus_di`, `di_diff`, `adx_chg_5` |
| volatility | `atr_pct`, `rv_21`, `rv_ratio_21_63` |
| oscillators | `rsi_14`, `macd`, `macd_signal`, `macd_hist` |
| volume | `vol_trend_20_60`, `vol_rel_5_20` |
| drawdown | `drawdown_63`, `drawdown_252` |
| relative | `rel_ret_21`, `rel_ret_63`, `bench_px_vs_sma200` |
| persistence (trend-specific) | `efficiency_ratio_21`, `efficiency_ratio_63`, `trend_persistence_63`, `trend_age` |
| position (state, not market) | `position_fraction`, `unrealized_pnl`, `holding_duration`, `stop_distance_atr` |
| trader (optional, static) | `risk_tolerance`, `drawdown_tolerance`, `horizon_preference`, `trade_frequency_preference` |

In [ ]:
FEATURE_REGISTRY = [  # (name, group, description)
    ("ret_1", "returns", "1-day log return"),
    ("ret_5", "returns", "5-day log return"),
    ("ret_21", "returns", "21-day log return"),
    ("ret_63", "returns", "63-day log return"),
    ("ret_126", "returns", "126-day log return"),
    ("px_vs_sma20", "ma_structure", "log(close/SMA20)"),
    ("px_vs_sma50", "ma_structure", "log(close/SMA50)"),
    ("px_vs_sma200", "ma_structure", "log(close/SMA200)"),
    ("sma20_vs_sma50", "ma_structure", "log(SMA20/SMA50)"),
    ("sma50_vs_sma200", "ma_structure", "log(SMA50/SMA200)"),
    ("ema12_vs_ema26", "ema", "log(EMA12/EMA26) crossover state"),
    ("ema20_vs_ema50", "ema", "log(EMA20/EMA50) crossover state"),
    ("sma50_slope_10", "slopes", "10-bar % change of SMA50"),
    ("sma200_slope_20", "slopes", "20-bar % change of SMA200"),
    ("ema20_slope_5", "slopes", "5-bar % change of EMA20"),
    ("adx", "trend_strength", "ADX(14)/100"),
    ("plus_di", "trend_strength", "+DI(14)/100"),
    ("minus_di", "trend_strength", "-DI(14)/100"),
    ("di_diff", "trend_strength", "(+DI - -DI)/100"),
    ("adx_chg_5", "trend_strength", "5-bar change of ADX /100"),
    ("atr_pct", "volatility", "ATR(14)/close"),
    ("rv_21", "volatility", "21-day realized vol (annualized)"),
    ("rv_ratio_21_63", "volatility", "log(rv21/rv63)"),
    ("rsi_14", "oscillators", "(RSI14-50)/50"),
    ("macd", "oscillators", "MACD(12,26)/close"),
    ("macd_signal", "oscillators", "MACD signal(9)/close"),
    ("macd_hist", "oscillators", "MACD histogram/close"),
    ("vol_trend_20_60", "volume", "log((volSMA20+1)/(volSMA60+1))"),
    ("vol_rel_5_20", "volume", "log((volSMA5+1)/(volSMA20+1))"),
    ("drawdown_63", "drawdown", "close / rolling-63 max - 1"),
    ("drawdown_252", "drawdown", "close / rolling-252 max - 1"),
    ("rel_ret_21", "relative", "21d log return minus benchmark 21d log return"),
    ("rel_ret_63", "relative", "63d log return minus benchmark 63d log return"),
    ("bench_px_vs_sma200", "relative", "log(benchmark close / benchmark SMA200)"),
    ("efficiency_ratio_21", "persistence", "Kaufman efficiency ratio, 21 bars (|net move| / path length)"),
    ("efficiency_ratio_63", "persistence", "Kaufman efficiency ratio, 63 bars"),
    ("trend_persistence_63", "persistence", "fraction of last 63 closes above SMA50"),
    ("trend_age", "persistence", "consecutive bars with close > SMA200, capped at 252, /252"),
]
MARKET_COLUMNS = [r[0] for r in FEATURE_REGISTRY]
POSITION_REGISTRY = [
    ("position_fraction", "position", "current units / max_units (0..1)"),
    ("unrealized_pnl", "position", "close / average entry price - 1, clipped to [-1, 1]; 0 when flat"),
    ("holding_duration", "position", "bars held since opening, capped at 252, /252"),
    ("stop_distance_atr", "position", "(peak close since entry - close)/ATR, capped at 10, /10"),
]
TRADER_COLUMNS = list(CFG["trader_profile"].keys()) if CFG["use_trader_features"] else []
FEATURE_COLUMNS = MARKET_COLUMNS + POSITION_COLUMNS + TRADER_COLUMNS
assert [r[0] for r in POSITION_REGISTRY] == POSITION_COLUMNS
N_MARKET, N_POS, N_TRADER = len(MARKET_COLUMNS), len(POSITION_COLUMNS), len(TRADER_COLUMNS)
POS_SLICE = slice(N_MARKET, N_MARKET + N_POS)
SCHEMA_HASH = hashlib.sha256("|".join(FEATURE_COLUMNS).encode()).hexdigest()[:16]
print(f"{len(FEATURE_COLUMNS)} features ({N_MARKET} market + {N_POS} position + {N_TRADER} trader) | schema hash {SCHEMA_HASH}")

In [ ]:
def ema(s, span):
    return s.ewm(span=span, adjust=False, min_periods=span).mean()

def wilder(s, n):
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()

def true_range(h, l, c):
    pc = c.shift(1)
    return pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)

def adx_di(h, l, c, n=14):
    up, dn = h.diff(), -l.diff()
    plus_dm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=c.index)
    minus_dm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=c.index)
    atr = wilder(true_range(h, l, c), n)
    plus_di = 100.0 * wilder(plus_dm, n) / atr
    minus_di = 100.0 * wilder(minus_dm, n) / atr
    denom = (plus_di + minus_di).replace(0, np.nan)
    dx = 100.0 * (plus_di - minus_di).abs() / denom
    return wilder(dx, n), plus_di, minus_di, atr

def rsi(c, n=14):
    d = c.diff()
    ru, rd = wilder(d.clip(lower=0), n), wilder((-d).clip(lower=0), n)
    return (100.0 * ru / (ru + rd).replace(0, np.nan)).fillna(50.0)

def run_length(flag):
    f = np.asarray(flag, dtype=bool); out = np.zeros(len(f)); run = 0
    for i, x in enumerate(f):
        run = run + 1 if x else 0
        out[i] = run
    return out

def efficiency_ratio(c, n):
    net = (c - c.shift(n)).abs()
    path = c.diff().abs().rolling(n).sum().replace(0, np.nan)
    return net / path

def compute_regime(sig, cfg):
    """Past-only regime classification. Uses only information available at the close of each bar.
    TRENDING  : supported up-trend (price > SMA200, SMA50 > SMA200, EMA20 > EMA50, SMA50 rising, ADX >= adx_trend)
    WEAKENING : not trending now, but TRENDING at some point in the last `recent_trend_window` bars
                (covers fading trends, pullbacks and reversals/breakdowns after an up-trend)
    SIDEWAYS  : everything else (low-ADX range, or no recent up-trend; long-only policy treats bear drifts as 'no trend')"""
    up_ok = ((sig["close"] > sig["sma_long"]) & (sig["sma_med"] > sig["sma_long"]) &
             (sig["ema_fast"] > sig["ema_slow"]) & (sig["slope_med"] > 0) & (sig["adx_raw"] >= cfg["adx_trend"]))
    recent = up_ok.astype(float).rolling(cfg["recent_trend_window"], min_periods=1).max().astype(bool)
    reg = np.where(up_ok, TRENDING, np.where(recent, WEAKENING, SIDEWAYS))
    return pd.Series(reg, index=sig.index, dtype=int)

def build_features(df, bench_close, cfg):
    """Returns (market_features[MARKET_COLUMNS], signals) after dropping the warm-up period. 100% backward-looking."""
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    lc = np.log(c); r1 = lc.diff()
    sma20, sma50, sma200 = c.rolling(20).mean(), c.rolling(50).mean(), c.rolling(200).mean()
    ema12, ema26, ema20, ema50 = ema(c, 12), ema(c, 26), ema(c, 20), ema(c, 50)
    adx_raw, pdi, mdi, atr = adx_di(h, l, c, 14)
    macd = ema12 - ema26; macd_sig = ema(macd.dropna(), 9).reindex(c.index); macd_hist = macd - macd_sig
    rv21, rv63 = r1.rolling(21).std() * np.sqrt(252), r1.rolling(63).std() * np.sqrt(252)

    b = bench_close.reindex(c.index).ffill()           # forward-fill only (causal)
    blc = np.log(b)
    bsma200 = bench_close.rolling(200).mean().reindex(c.index).ffill()

    f = pd.DataFrame(index=c.index)
    for n in (1, 5, 21, 63, 126):
        f[f"ret_{n}"] = lc.diff(n)
    f["px_vs_sma20"], f["px_vs_sma50"], f["px_vs_sma200"] = np.log(c / sma20), np.log(c / sma50), np.log(c / sma200)
    f["sma20_vs_sma50"], f["sma50_vs_sma200"] = np.log(sma20 / sma50), np.log(sma50 / sma200)
    f["ema12_vs_ema26"], f["ema20_vs_ema50"] = np.log(ema12 / ema26), np.log(ema20 / ema50)
    f["sma50_slope_10"], f["sma200_slope_20"], f["ema20_slope_5"] = sma50.pct_change(10), sma200.pct_change(20), ema20.pct_change(5)
    f["adx"], f["plus_di"], f["minus_di"] = adx_raw / 100, pdi / 100, mdi / 100
    f["di_diff"], f["adx_chg_5"] = (pdi - mdi) / 100, adx_raw.diff(5) / 100
    f["atr_pct"], f["rv_21"], f["rv_ratio_21_63"] = atr / c, rv21, np.log(rv21 / rv63)
    f["rsi_14"] = (rsi(c, 14) - 50) / 50
    f["macd"], f["macd_signal"], f["macd_hist"] = macd / c, macd_sig / c, macd_hist / c
    f["vol_trend_20_60"] = np.log((v.rolling(20).mean() + 1) / (v.rolling(60).mean() + 1))
    f["vol_rel_5_20"] = np.log((v.rolling(5).mean() + 1) / (v.rolling(20).mean() + 1))
    f["drawdown_63"], f["drawdown_252"] = c / c.rolling(63).max() - 1, c / c.rolling(252).max() - 1
    f["rel_ret_21"], f["rel_ret_63"] = lc.diff(21) - blc.diff(21), lc.diff(63) - blc.diff(63)
    f["bench_px_vs_sma200"] = np.log(b / bsma200)
    f["efficiency_ratio_21"], f["efficiency_ratio_63"] = efficiency_ratio(c, 21), efficiency_ratio(c, 63)
    f["trend_persistence_63"] = (c > sma50).astype(float).where(sma50.notna()).rolling(63).mean()
    f["trend_age"] = pd.Series(np.minimum(run_length((c > sma200).fillna(False).values), 252) / 252.0, index=c.index)
    f = f[MARKET_COLUMNS].replace([np.inf, -np.inf], np.nan)

    sig = pd.DataFrame({"open": o, "close": c, "sma_med": sma50, "sma_long": sma200, "ema_fast": ema20, "ema_slow": ema50,
                        "adx_raw": adx_raw, "adx_chg5_raw": adx_raw.diff(5), "macd_hist_raw": macd_hist, "atr": atr,
                        "slope_med": sma50.pct_change(10)}, index=c.index)
    sig["regime"] = compute_regime(sig, cfg)            # computed on the full history *before* slicing => no warm-up artefacts

    valid = f.notna().all(axis=1) & sig[["open", "close", "sma_med", "sma_long", "atr"]].notna().all(axis=1)
    if not valid.any():
        raise ValueError("no fully valid feature row")
    first = int(np.argmax(valid.values))
    f, sig = f.iloc[first:], sig.iloc[first:]
    n_missing = int(f.isna().sum().sum())
    if n_missing:                                       # interior NaNs: causal forward fill, then report
        print(f"   note: {n_missing} interior NaNs forward-filled")
        f = f.ffill().fillna(0.0); sig = sig.ffill()
    return f, sig

# --- look-ahead unit test: features computed on truncated data must equal features computed on full data ---
_t = TRAIN_TICKERS[0]; _df = RAW[_t]; _cut = int(len(_df) * 0.7)
_full, _ = build_features(_df, RAW[CFG["benchmark"]]["Close"], CFG)
_part, _ = build_features(_df.iloc[:_cut], RAW[CFG["benchmark"]]["Close"].loc[:_df.index[_cut - 1]], CFG)
_common = _part.index.intersection(_full.index)[-250:]
_diff = (_part.loc[_common] - _full.loc[_common]).abs().max().max()
assert _diff < 1e-8, f"LOOK-AHEAD DETECTED: max abs diff {_diff}"
print(f"Look-ahead test passed on {_t}: max |truncated - full| = {_diff:.2e} over {len(_common)} bars")

## 6. Position tracker, regimes and action labels

### Regimes (past-only; see `compute_regime`)
`TRENDING` (supported up-trend) · `WEAKENING` (trend lost/fading within the last 40 bars) · `SIDEWAYS` (no supported/recent trend).

### Proxy-label rule — the "teacher" (documented, causal)
Evaluated at each close, **given the current position**:

| Position | Condition | Action |
|---|---|---|
| flat | regime = TRENDING and cool-down over | `BUY` (1 unit) |
| flat | otherwise | `HOLD` |
| long | close < SMA200, **or** (close < SMA50 and EMA20 < EMA50), **or** close < peak close since entry − 3·ATR | `EXIT` (+ cool-down) |
| long, >1 unit | regime = WEAKENING | `SELL` (trim one unit) |
| long, 1 unit | WEAKENING for ≥ 10 consecutive bars | `EXIT` |
| long, < max units | TRENDING, ADX ≥ 30, held ≥ 10 bars, unrealized P&L > 0 | `BUY` (scale in) |
| long | otherwise | `HOLD` |

The same `PositionTracker` class drives labeling **and** backtesting, so action semantics are identical in both. Invalid actions (e.g. EXIT while flat) are stored as `HOLD`.

**Important caveats:** the rule only uses information the network also sees, so the network can in principle learn it almost exactly. Labels are generated by simulating the rule's own trade path; in the backtest the network's *own* position drives the position features (a distribution shift that is deliberately part of the test).

In [ ]:
class PositionTracker:
    """Long-only discrete-unit position state. Shared by labeler and backtester."""
    def __init__(self, max_units=2, buy_step=1, sell_step=1):
        self.max_units, self.buy_step, self.sell_step = int(max_units), int(buy_step), int(sell_step)
        self.reset()

    def reset(self):
        self.units, self.avg_entry, self.peak_close, self.bars_held = 0, np.nan, np.nan, 0

    def effective_action(self, a):
        if a == BUY:  return BUY if self.units < self.max_units else HOLD
        if a == SELL: return SELL if self.units > 0 else HOLD
        if a == EXIT: return EXIT if self.units > 0 else HOLD
        return HOLD

    def target_units(self, a):
        a = self.effective_action(a)
        if a == BUY:  return min(self.max_units, self.units + self.buy_step)
        if a == SELL: return max(0, self.units - self.sell_step)
        if a == EXIT: return 0
        return self.units

    def apply_fill(self, a, fill_price):
        a = self.effective_action(a); old = self.units; new = self.target_units(a)
        if a == BUY:
            if old == 0:
                self.avg_entry, self.peak_close, self.bars_held = fill_price, fill_price, 0
            else:
                self.avg_entry = (self.avg_entry * old + fill_price * (new - old)) / new
        self.units = new
        if new == 0:
            self.avg_entry, self.peak_close, self.bars_held = np.nan, np.nan, 0
        return old, new

    def mark(self, close):
        if self.units > 0:
            self.bars_held += 1
            self.peak_close = max(self.peak_close, close)

    def unrealized(self, close):
        return float(np.clip(close / self.avg_entry - 1.0, -1.0, 1.0)) if self.units > 0 else 0.0

    def features(self, close, atr):
        if self.units == 0:
            return np.zeros(N_POS, dtype=np.float64)
        stop_dist = min(max((self.peak_close - close) / max(atr, 1e-12), 0.0), 10.0) / 10.0
        return np.array([self.units / self.max_units, self.unrealized(close),
                         min(self.bars_held, 252) / 252.0, stop_dist], dtype=np.float64)


class TrendRule:
    """Causal trend-following teacher (documented above). One instance per asset run."""
    def __init__(self, cfg):
        self.c = cfg; self.reset()
    def reset(self):
        self.cooldown, self.weak_streak = 0, 0
    def decide(self, S, i, regime, trk):
        c, atr = S["close"][i], S["atr"][i]
        if self.cooldown > 0:
            self.cooldown -= 1
        if trk.units == 0:
            self.weak_streak = 0
            return BUY if (regime == TRENDING and self.cooldown == 0) else HOLD
        self.weak_streak = self.weak_streak + 1 if regime == WEAKENING else 0
        stop = trk.peak_close - self.c["atr_stop_mult"] * atr
        broken = (c < S["sma_long"][i]) or (c < S["sma_med"][i] and S["ema_fast"][i] < S["ema_slow"][i]) or (c < stop)
        if broken or (regime == WEAKENING and trk.units == 1 and self.weak_streak >= self.c["weak_exit_bars"]):
            self.cooldown = self.c["cooldown_bars"]
            return EXIT
        if regime == WEAKENING and trk.units > 1:
            return SELL
        if (regime == TRENDING and trk.units < trk.max_units and S["adx_raw"][i] >= self.c["adx_add"]
                and trk.bars_held >= self.c["min_hold_add"] and trk.unrealized(c) > 0):
            return BUY
        return HOLD


def signals_to_arrays(sig):
    return {k: sig[k].to_numpy(dtype=float) for k in sig.columns if k != "regime"} | {"regime": sig["regime"].to_numpy(dtype=int)}


def load_human_actions(path):
    """CSV columns: date, ticker, action (0-3 or HOLD/BUY/SELL/EXIT). Returns {ticker: {Timestamp: action}}."""
    df = pd.read_csv(path, parse_dates=["date"])
    name2id = {v: k for k, v in ACTIONS.items()}
    df["action"] = df["action"].map(lambda x: name2id[str(x).upper()] if str(x).upper() in name2id else int(x))
    assert df["action"].isin(ACTIONS.keys()).all(), "actions must be 0..3"
    out = {}
    for t, g in df.groupby("ticker"):
        out[t] = dict(zip(pd.to_datetime(g["date"]).dt.normalize(), g["action"]))
    return out


def generate_labels(sig, cfg, human=None):
    """Runs the teacher rule (or applies human actions where supplied) and returns
    labels, position features (state at decision time), label_source (0=proxy, 1=human)."""
    S = signals_to_arrays(sig); n = len(sig); dates = sig.index
    trk, rule = PositionTracker(cfg["max_units"]), TrendRule(cfg)
    y = np.zeros(n, dtype=np.int64); pos = np.zeros((n, N_POS)); src = np.zeros(n, dtype=np.int8)
    pending = HOLD
    for i in range(n):
        if i > 0:
            trk.apply_fill(pending, S["open"][i])        # action decided at i-1 fills at open of i
        trk.mark(S["close"][i])
        pos[i] = trk.features(S["close"][i], S["atr"][i])
        a = rule.decide(S, i, S["regime"][i], trk)       # always called so teacher state stays updated
        if human is not None and dates[i] in human:
            a, src[i] = human[dates[i]], 1
        y[i] = trk.effective_action(a)
        pending = y[i]
    return y, pos, src

HUMAN = load_human_actions(CFG["human_actions_csv"]) if CFG["human_actions_csv"] else None
LABEL_MODE = "human_override_where_available" if HUMAN else "proxy_rule_only"
print("Label mode:", LABEL_MODE)
if not HUMAN:
    print("NOTE: all labels are PROXY labels from the trend-following rule (not human demonstrations).")

## 7. Build the panel dataset (all tickers) with chronological splits
Scaling statistics are fitted on **training-period rows only**. Windows are built per ticker so no sequence crosses assets.

In [ ]:
PANEL, skipped = {}, []
bench_close = RAW[CFG["benchmark"]]["Close"]
for t in TRAIN_TICKERS:
    feat, sig = build_features(RAW[t], bench_close, CFG)
    if len(feat) < CFG["min_rows_after_warmup"]:
        skipped.append((t, len(feat))); continue
    y, pos, src = generate_labels(sig, CFG, HUMAN.get(t) if HUMAN else None)
    PANEL[t] = dict(dates=feat.index, feat=feat, sig=sig, y=y, pos=pos, src=src, regime=sig["regime"].to_numpy(int))
if skipped:
    print("!!! Skipped (too few rows after feature warm-up):", skipped)
ASSETS = list(PANEL.keys())
assert len(ASSETS) >= CFG["min_assets"], "too few assets after warm-up filtering"

# global (concatenated) arrays
SL, parts_m, parts_p, parts_y, parts_r, parts_d, parts_t, parts_s = {}, [], [], [], [], [], [], []
cur = 0
for k, t in enumerate(ASSETS):
    p = PANEL[t]; n = len(p["dates"]); SL[t] = (cur, cur + n); cur += n
    parts_m.append(p["feat"].to_numpy(float)); parts_p.append(p["pos"]); parts_y.append(p["y"]); parts_r.append(p["regime"])
    parts_d.append(p["dates"]); parts_t.append(np.full(n, k)); parts_s.append(p["src"])
X_MARKET_RAW = np.vstack(parts_m); X_POS = np.vstack(parts_p)
Y_ALL, R_ALL, S_ALL, T_ALL = map(np.concatenate, (parts_y, parts_r, parts_s, parts_t))
D_ALL = parts_d[0].append(parts_d[1:]) if len(parts_d) > 1 else parts_d[0]
TRADER_VALUES = np.array([CFG["trader_profile"][c] for c in TRADER_COLUMNS], dtype=float)
X_TRADER = np.tile(TRADER_VALUES, (len(Y_ALL), 1)) if N_TRADER else np.zeros((len(Y_ALL), 0))

# sample = decision bar (window ends at this row); needs seq_len rows inside the same ticker
L = CFG["seq_len"]
IDX = {"train": [], "val": [], "test": []}
tr_end, va_end = pd.Timestamp(CFG["train_end"]), pd.Timestamp(CFG["val_end"])
for t in ASSETS:
    a, b = SL[t]
    for e in range(a + L - 1, b):
        d = D_ALL[e]
        IDX["train" if d <= tr_end else "val" if d <= va_end else "test"].append(e)
IDX = {k: np.asarray(v, dtype=np.int64) for k, v in IDX.items()}
for k, v in IDX.items():
    assert len(v) > 0, f"split '{k}' is empty - adjust train_end/val_end/start/end"
    print(f"{k:5s}: {len(v):6d} samples  {D_ALL[v].min().date()} -> {D_ALL[v].max().date()}")
assert D_ALL[IDX["train"]].max() < D_ALL[IDX["val"]].min() and D_ALL[IDX["val"]].max() < D_ALL[IDX["test"]].min()

# scaler fitted ONLY on training-period market features
fit_rows = np.where(np.asarray(D_ALL <= tr_end))[0]
SCALER = StandardScaler().fit(X_MARKET_RAW[fit_rows])
X_MARKET = np.clip(SCALER.transform(X_MARKET_RAW), -CFG["feature_clip"], CFG["feature_clip"])
X_ALL = np.hstack([X_MARKET, X_POS, X_TRADER]).astype(np.float32)
assert X_ALL.shape[1] == len(FEATURE_COLUMNS) and np.isfinite(X_ALL).all()
print("X_ALL:", X_ALL.shape, "| scaler fitted on", len(fit_rows), "training-period rows")

# label / regime composition
lab_tbl = pd.DataFrame({sp: pd.Series(Y_ALL[IDX[sp]]).map(ACTIONS).value_counts(normalize=True) for sp in IDX}).fillna(0).round(4)
reg_tbl = pd.DataFrame({sp: pd.Series(R_ALL[IDX[sp]]).map(REGIMES).value_counts(normalize=True) for sp in IDX}).fillna(0).round(4)
print("\nAction label distribution by split:\n", lab_tbl, "\n\nRegime distribution by split:\n", reg_tbl)
print("\nHuman-labelled share of samples:", {sp: float(S_ALL[IDX[sp]].mean().round(4)) for sp in IDX})

# --- distinctness diagnostic vs a simple momentum proxy (acceleration of short vs medium returns) ---
mom_accel = pd.Series(X_MARKET_RAW[:, MARKET_COLUMNS.index("ret_5")] - X_MARKET_RAW[:, MARKET_COLUMNS.index("ret_21")] / 4.2)
trend_buy = pd.Series((Y_ALL == BUY).astype(float))
print(f"\nDistinctness check: corr(BUY-label, short-vs-medium return acceleration) = {trend_buy.corr(mom_accel):.3f}  (low |corr| => not just a momentum rule)")

## 8. Sequence dataset
Windows are sliced lazily from one big float32 tensor (low memory). A sample's window covers rows `[e-seq_len+1, e]` of a single ticker; the label is the action at row `e`.

*Optional real trader actions:* set `CFG["human_actions_csv"]` to a CSV with columns `date,ticker,action` (action = 0–3 or HOLD/BUY/SELL/EXIT) **before** running section 6. Human actions override the proxy rule on matching dates and are flagged in `S_ALL`; all other dates remain proxy labels. For a clean human-demonstration dataset, supply a complete action log.

In [ ]:
X_T = torch.from_numpy(X_ALL)
Y_T = torch.from_numpy(Y_ALL.astype(np.int64))

class SeqDataset(Dataset):
    def __init__(self, X, Y, idx, seq_len):
        self.X, self.Y, self.idx, self.L = X, Y, idx, seq_len
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        e = int(self.idx[k])
        return self.X[e - self.L + 1: e + 1], self.Y[e]

DS = {sp: SeqDataset(X_T, Y_T, IDX[sp], L) for sp in IDX}
_g = torch.Generator(); _g.manual_seed(CFG["seed"])
LOADERS = {
    "train": DataLoader(DS["train"], batch_size=CFG["batch_size"], shuffle=True, generator=_g, num_workers=0, drop_last=False),
    "val": DataLoader(DS["val"], batch_size=1024, shuffle=False, num_workers=0),
    "test": DataLoader(DS["test"], batch_size=1024, shuffle=False, num_workers=0),
}
xb, yb = next(iter(LOADERS["train"])); print("batch:", tuple(xb.shape), tuple(yb.shape))
# NOTE: shuffling is only applied inside the TRAIN split (all train samples precede validation). Splits themselves are chronological.

## 9. Model and policy class
`TrendGRUNet`: linear embedding → GRU/LSTM → LayerNorm → MLP head with **4 action logits** + a **confidence head**.
The confidence head is trained (BCE) to predict *whether the action head's argmax matches the label*; it sees detached features, so it never distorts the action head. It is a learned reliability score, **not** a Bayesian probability.

`TrendFollowingPolicy` is the inference wrapper required by the project: `predict(state)`, `save(path)`, `load(path, device)`. It stores its own normalization statistics, so predicting needs **no retraining, no data download, and no sklearn pickle**.

In [ ]:
class TrendGRUNet(nn.Module):
    def __init__(self, input_dim, hidden_size=64, num_layers=2, rnn_type="GRU", dropout=0.2, n_actions=4):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(input_dim, hidden_size), nn.GELU())
        rnn_cls = {"GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type.upper()]
        self.rnn = rnn_cls(hidden_size, hidden_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_size, n_actions))
        self.conf_head = nn.Sequential(nn.Linear(hidden_size + n_actions, 32), nn.GELU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.rnn(self.embed(x))
        z = self.norm(out[:, -1])
        logits = self.head(z)
        conf_logit = self.conf_head(torch.cat([z.detach(), F.softmax(logits, -1).detach()], dim=-1)).squeeze(-1)
        return logits, conf_logit


def _jsonable(o):
    return json.loads(json.dumps(o, default=lambda x: x.item() if hasattr(x, "item") else str(x)))


class TrendFollowingPolicy:
    """SP-03 inference wrapper. Action convention: 0 HOLD, 1 BUY, 2 SELL (trim one unit), 3 EXIT (close all)."""
    ACTIONS = {0: "HOLD", 1: "BUY", 2: "SELL", 3: "EXIT"}

    def __init__(self, model, meta, device="cpu"):
        self.device = torch.device(device)
        self.model = model.to(self.device).eval()
        self.meta = meta
        self.feature_columns = list(meta["feature_columns"])
        self.market_idx = [self.feature_columns.index(c) for c in meta["market_columns"]]
        self.pos_frac_idx = self.feature_columns.index("position_fraction")
        self.seq_len = int(meta["seq_len"]); self.max_units = int(meta["max_units"])
        self.mean = np.asarray(meta["scaler_mean"], dtype=np.float64)
        self.scale = np.asarray(meta["scaler_scale"], dtype=np.float64)
        self.clip = float(meta["feature_clip"])

    # ---------- helpers ----------
    def _to_array(self, state):
        if isinstance(state, dict):
            state = state.get("features", state.get("state"))
        if isinstance(state, pd.DataFrame):
            missing = [c for c in self.feature_columns if c not in state.columns]
            if missing:
                raise ValueError(f"state is missing feature columns: {missing}")
            arr = state[self.feature_columns].to_numpy(dtype=np.float64)
        elif isinstance(state, torch.Tensor):
            arr = state.detach().cpu().numpy().astype(np.float64)
        else:
            arr = np.asarray(state, dtype=np.float64)
        if arr.ndim == 3 and arr.shape[0] == 1:
            arr = arr[0]
        if arr.ndim != 2 or arr.shape[1] != len(self.feature_columns):
            raise ValueError(f"state must have shape (T, {len(self.feature_columns)}) in schema order; got {arr.shape}")
        if arr.shape[0] < self.seq_len:
            raise ValueError(f"need at least seq_len={self.seq_len} rows; got {arr.shape[0]}")
        arr = arr[-self.seq_len:].copy()
        if not np.isfinite(arr).all():
            raise ValueError("state contains NaN/inf")
        return arr

    def normalize(self, arr):
        out = arr.copy()
        out[:, self.market_idx] = np.clip((out[:, self.market_idx] - self.mean) / self.scale, -self.clip, self.clip)
        return out

    def valid_action_mask(self, position_fraction):
        units = int(round(float(position_fraction) * self.max_units))
        return np.array([1.0, float(units < self.max_units), float(units > 0), float(units > 0)])

    # ---------- public API ----------
    @torch.no_grad()
    def predict(self, state, normalized=False, mask_invalid=True):
        """state: array/DataFrame/tensor (T>=seq_len, n_features) in schema order. Raw (unscaled) market features by default;
        pass normalized=True if market features are already standardized+clipped with this policy's statistics.
        Position columns are used as given (already bounded). Returns dict with the 4 action probabilities, predicted action, confidence."""
        arr = self._to_array(state)
        mask = self.valid_action_mask(arr[-1, self.pos_frac_idx])
        x = arr if normalized else self.normalize(arr)
        xt = torch.as_tensor(x, dtype=torch.float32, device=self.device).unsqueeze(0)
        logits, conf_logit = self.model(xt)
        probs = torch.softmax(logits, -1)[0].cpu().numpy().astype(np.float64)
        conf = float(torch.sigmoid(conf_logit)[0].cpu())
        masked = probs * mask if mask_invalid else probs
        a = int(np.argmax(masked))
        return {
            "policy_id": self.meta["policy_id"],
            "action_probabilities": probs.tolist(),
            "action_probabilities_named": {self.ACTIONS[i]: float(probs[i]) for i in range(4)},
            "predicted_action": a,
            "action_name": self.ACTIONS[a],
            "confidence": conf,                       # learned reliability of the action head (0..1)
            "max_probability": float(probs.max()),
            "raw_argmax_action": int(np.argmax(probs)),
            "valid_action_mask": mask.tolist(),
        }

    def save(self, path):
        ckpt = {"format_version": 1,
                "model_state": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
                "meta": _jsonable(self.meta)}
        torch.save(ckpt, path)

    @classmethod
    def load(cls, path, device=None):
        device = torch.device(device) if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(path, map_location=device, weights_only=True)
        meta = ckpt["meta"]
        model = TrendGRUNet(**meta["model_args"])
        model.load_state_dict(ckpt["model_state"])
        return cls(model, meta, device)


def make_meta(model_args, scaler):
    return {
        "policy_id": CFG["policy_id"], "policy_name": CFG["policy_name"], "format_version": 1,
        "model_args": model_args, "feature_columns": FEATURE_COLUMNS, "market_columns": MARKET_COLUMNS,
        "position_columns": POSITION_COLUMNS, "trader_columns": TRADER_COLUMNS, "schema_hash": SCHEMA_HASH,
        "scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist(), "feature_clip": CFG["feature_clip"],
        "seq_len": CFG["seq_len"], "max_units": CFG["max_units"], "action_map": {str(k): v for k, v in ACTIONS.items()},
        "regime_map": {str(k): v for k, v in REGIMES.items()},
        "label_mode": LABEL_MODE, "trained_on_tickers": ASSETS,
        "train_end": CFG["train_end"], "val_end": CFG["val_end"],
    }

MODEL_ARGS = dict(input_dim=len(FEATURE_COLUMNS), hidden_size=CFG["hidden_size"], num_layers=CFG["num_layers"],
                  rnn_type=CFG["rnn_type"], dropout=CFG["dropout"], n_actions=4)
set_seed(CFG["seed"])
MODEL = TrendGRUNet(**MODEL_ARGS).to(DEVICE)
print(MODEL); print("trainable parameters:", sum(p.numel() for p in MODEL.parameters() if p.requires_grad))

## 10. Training
* Class weights from the **training labels only** (`balanced_sqrt` dampens extreme weights; HOLD usually dominates, SELL is rare).
* Early stopping on validation loss with best-weights restore; LR reduced on plateau.
* Seeds, deterministic cuDNN and a seeded DataLoader generator for reproducibility (exact bitwise reproducibility across different GPUs/driver versions is not guaranteed).

In [ ]:
def make_class_weights(y_train, mode):
    w = np.ones(4)
    present = np.unique(y_train)
    if mode != "none":
        bal = compute_class_weight("balanced", classes=present, y=y_train)
        if mode == "balanced_sqrt":
            bal = np.sqrt(bal)
        w[present] = bal / bal.mean()
    counts = np.bincount(y_train, minlength=4)
    rare = {ACTIONS[c]: int(counts[c]) for c in range(4) if 0 < counts[c] < 200}
    if rare:
        print(f"!!! WARNING: very rare classes in training labels (count): {rare} - expect noisy precision/recall for them")
    missing = [ACTIONS[c] for c in range(4) if c not in present]
    if missing:
        print(f"!!! WARNING: classes absent from training labels: {missing} (the model cannot learn them)")
    return torch.tensor(w, dtype=torch.float32)

CLASS_W = make_class_weights(Y_ALL[IDX["train"]], CFG["class_weight_mode"])
print("class weights:", {ACTIONS[i]: round(float(CLASS_W[i]), 3) for i in range(4)})
CRIT = nn.CrossEntropyLoss(weight=CLASS_W.to(DEVICE), label_smoothing=CFG["label_smoothing"])

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    tot, n, preds, ys = 0.0, 0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits, conf_logit = model(xb)
            loss_cls = CRIT(logits, yb)
            correct = (logits.argmax(1) == yb).float().detach()
            loss = loss_cls + CFG["conf_loss_weight"] * F.binary_cross_entropy_with_logits(conf_logit, correct)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                optimizer.step()
        tot += loss_cls.item() * len(yb); n += len(yb)
        preds.append(logits.argmax(1).cpu().numpy()); ys.append(yb.cpu().numpy())
    p, y = np.concatenate(preds), np.concatenate(ys)
    return tot / n, accuracy_score(y, p), f1_score(y, p, average="macro", labels=[0, 1, 2, 3], zero_division=0)

optimizer = torch.optim.AdamW(MODEL.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
HIST = {k: [] for k in ["epoch", "train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1", "lr"]}
best_val, best_state, best_epoch, bad = float("inf"), None, -1, 0
t0 = time.time()
for epoch in range(1, CFG["max_epochs"] + 1):
    tr = run_epoch(MODEL, LOADERS["train"], optimizer)
    va = run_epoch(MODEL, LOADERS["val"])
    scheduler.step(va[0])
    for k, v in zip(HIST, [epoch, tr[0], va[0], tr[1], va[1], tr[2], va[2], optimizer.param_groups[0]["lr"]]):
        HIST[k].append(v)
    improved = va[0] < best_val - 1e-5
    if improved:
        best_val, best_epoch, bad = va[0], epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in MODEL.state_dict().items()}
    else:
        bad += 1
    print(f"ep {epoch:3d} | train loss {tr[0]:.4f} acc {tr[1]:.3f} f1 {tr[2]:.3f} | val loss {va[0]:.4f} acc {va[1]:.3f} f1 {va[2]:.3f} | lr {optimizer.param_groups[0]['lr']:.1e}{'  *' if improved else ''}")
    if bad >= CFG["patience"]:
        print(f"Early stopping at epoch {epoch} (best epoch {best_epoch})"); break
MODEL.load_state_dict(best_state); MODEL.eval()
TRAIN_SECONDS = time.time() - t0
print(f"Restored best weights from epoch {best_epoch} (val loss {best_val:.4f}); training took {TRAIN_SECONDS:.0f}s")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(HIST["epoch"], HIST["train_loss"], label="train"); ax[0].plot(HIST["epoch"], HIST["val_loss"], label="validation")
ax[0].axvline(best_epoch, color="gray", ls="--", label="best epoch"); ax[0].set_title("Weighted cross-entropy"); ax[0].legend()
ax[1].plot(HIST["epoch"], HIST["train_acc"], label="train"); ax[1].plot(HIST["epoch"], HIST["val_acc"], label="validation"); ax[1].set_title("Accuracy"); ax[1].legend()
ax[2].plot(HIST["epoch"], HIST["train_f1"], label="train"); ax[2].plot(HIST["epoch"], HIST["val_f1"], label="validation"); ax[2].set_title("Macro F1"); ax[2].legend()
for a_ in ax: a_.set_xlabel("epoch"); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 11. Classification evaluation (imitation fidelity vs. the labels)
Remember: with proxy labels these numbers say *how closely the network reproduces the rule*. Good scores here do **not** imply profitable trading.
Reported: per-class precision/recall/F1, confusion matrices, a majority-class baseline, how often the raw argmax is an *invalid* action for the current position, confidence-head calibration, and regime-specific results.

In [ ]:
@torch.no_grad()
def predict_split(model, split):
    model.eval(); P, C = [], []
    for xb, _ in LOADERS[split]:
        logits, conf_logit = model(xb.to(DEVICE))
        P.append(torch.softmax(logits, -1).cpu().numpy()); C.append(torch.sigmoid(conf_logit).cpu().numpy())
    return np.concatenate(P), np.concatenate(C)

EVAL = {}
for sp in ["val", "test"]:
    P, C = predict_split(MODEL, sp)
    idx = IDX[sp]; y = Y_ALL[idx]; pred = P.argmax(1)
    pos_frac = X_ALL[idx, N_MARKET]
    units = np.rint(pos_frac * CFG["max_units"]).astype(int)
    invalid = ((pred == BUY) & (units >= CFG["max_units"])) | (np.isin(pred, [SELL, EXIT]) & (units == 0))
    maj = np.bincount(Y_ALL[IDX["train"]], minlength=4).argmax()
    EVAL[sp] = dict(P=P, C=C, y=y, pred=pred, idx=idx, regime=R_ALL[idx], invalid_rate=float(invalid.mean()),
                    acc=float(accuracy_score(y, pred)), bal_acc=float(balanced_accuracy_score(y, pred)),
                    macro_f1=float(f1_score(y, pred, average="macro", labels=[0, 1, 2, 3], zero_division=0)),
                    majority_acc=float((y == maj).mean()))
    print(f"\n===== {sp.upper()} ({len(y)} samples) =====")
    print(f"accuracy {EVAL[sp]['acc']:.4f} | balanced acc {EVAL[sp]['bal_acc']:.4f} | macro F1 {EVAL[sp]['macro_f1']:.4f} | "
          f"majority-class ('{ACTIONS[maj]}') acc {EVAL[sp]['majority_acc']:.4f} | invalid-raw-action rate {EVAL[sp]['invalid_rate']:.4f}")
    print(classification_report(y, pred, labels=[0, 1, 2, 3], target_names=list(ACTIONS.values()), zero_division=0, digits=3))
    EVAL[sp]["report"] = classification_report(y, pred, labels=[0, 1, 2, 3], target_names=list(ACTIONS.values()), zero_division=0, output_dict=True)
    EVAL[sp]["cm"] = confusion_matrix(y, pred, labels=[0, 1, 2, 3])

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for r_, sp in enumerate(["val", "test"]):
    cm = EVAL[sp]["cm"]; cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    for c_, (m, ttl) in enumerate([(cm, "counts"), (cmn, "row-normalized")]):
        ax = axes[r_, c_]; im = ax.imshow(m, cmap="Blues"); ax.set_title(f"{sp} confusion ({ttl})")
        ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(ACTIONS.values()); ax.set_yticklabels(ACTIONS.values())
        ax.set_xlabel("predicted"); ax.set_ylabel("label")
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{m[i, j]:.2f}" if ttl != "counts" else int(m[i, j]), ha="center", va="center",
                        color="white" if m[i, j] > m.max() * 0.6 else "black", fontsize=9)
plt.tight_layout(); plt.show()

# confidence-head calibration (test)
e = EVAL["test"]; correct = (e["pred"] == e["y"]).astype(float)
q = pd.qcut(pd.Series(e["C"]), 5, duplicates="drop")
CAL = pd.DataFrame({"conf": e["C"], "correct": correct, "bin": q}).groupby("bin", observed=True).agg(n=("correct", "size"), mean_conf=("conf", "mean"), accuracy=("correct", "mean"))
print("\nConfidence-head calibration on TEST (quintiles):\n", CAL.round(4))

In [ ]:
# Regime-specific evaluation
def regime_table(sp):
    e = EVAL[sp]; rows = []
    for rid, rname in REGIMES.items():
        m = e["regime"] == rid
        if m.sum() == 0:
            rows.append(dict(split=sp, regime=rname, n=0)); continue
        y, p = e["y"][m], e["pred"][m]
        row = dict(split=sp, regime=rname, n=int(m.sum()), share=round(float(m.mean()), 3), accuracy=round(float(accuracy_score(y, p)), 4),
                   macro_f1=round(float(f1_score(y, p, average="macro", labels=[0, 1, 2, 3], zero_division=0)), 4),
                   mean_confidence=round(float(e["C"][m].mean()), 4))
        for a_, nm in ACTIONS.items():
            row[f"label_{nm}"] = round(float((y == a_).mean()), 3); row[f"pred_{nm}"] = round(float((p == a_).mean()), 3)
        rows.append(row)
    return pd.DataFrame(rows)

REGIME_EVAL = pd.concat([regime_table("val"), regime_table("test")], ignore_index=True)
print(REGIME_EVAL.to_string(index=False))

## 12. Backtest engine

**Timing (no look-ahead):** at the close of bar *t* the policy sees features through *t* → emits an action → the order fills at the **open of *t+1*** at `open × (1 ± slippage)` plus commission. The last bar's decision is discarded (it could not be executed). Equity is marked at the close.

**Sizing:** target exposure = `units / max_units × max_position_fraction × equity-at-open`. Long-only, no leverage, cash can't go negative (buys are capped by available cash incl. fees). Each asset runs as an independent account with `initial_capital / n_assets`; the portfolio is the sum (no cross-asset rebalancing).
**Invalid actions** (e.g. EXIT when flat) are executed as HOLD. Test-period backtests start **flat**.

**Compared strategies** (all pay the same costs, same timing, same start):
1. **SP-03 policy** (the trained network; its own position drives its position features)
2. **Buy & hold** (equal-weight across the same assets) and **benchmark buy & hold**
3. **Simple trend baseline:** long 100 % when SMA50 > SMA200, flat otherwise (one-step entry/exit)
4. **Rule teacher:** the proxy-label rule run live (shows how much of the policy's behaviour is just the rule)

In [ ]:
def build_pack(sym, with_X):
    if sym in PANEL:
        p = PANEL[sym]; a, b = SL[sym]
        sig, dates = p["sig"], p["dates"]
        X = X_ALL[a:b] if with_X else None
    else:
        feat, sig = build_features(RAW[sym], RAW[CFG["benchmark"]]["Close"], CFG); dates, X = feat.index, None
    return dict(sym=sym, dates=dates, S=signals_to_arrays(sig), X=X, n=len(dates), start=int(dates.searchsorted(TEST_START)))

PACKS = {t: build_pack(t, True) for t in ASSETS}
BENCH_PACK = build_pack(CFG["benchmark"], False)
for t, pk in PACKS.items():
    assert pk["start"] >= CFG["seq_len"] and pk["n"] - pk["start"] > 60, f"{t}: test period too short"
print("Test bars per asset:", {t: pk["n"] - pk["start"] for t, pk in PACKS.items()})


def run_backtest(pack, decider, buy_step=1, cost_mult=1.0, cap=None):
    S, n, dates = pack["S"], pack["n"], pack["dates"]
    cap = CFG["initial_capital"] / len(ASSETS) if cap is None else cap
    comm, slip = CFG["commission_bps"] * 1e-4 * cost_mult, CFG["slippage_bps"] * 1e-4 * cost_mult
    trk = PositionTracker(CFG["max_units"], buy_step=buy_step)
    pos_hist = np.zeros((n, N_POS))
    cash, shares, pending = cap, 0.0, HOLD
    rec, fills, cycles = [], [], []
    cyc_cf = cyc_cost = 0.0; cyc_start = None
    for i in range(pack["start"], n):
        o, c = S["open"][i], S["close"][i]
        if pending != HOLD:
            eff = trk.effective_action(pending)
            if eff != HOLD:
                old_u, new_u = trk.units, trk.target_units(eff)
                eq_open = cash + shares * o
                target = eq_open * CFG["max_position_fraction"] * new_u / trk.max_units / o
                delta = target - shares
                if new_u == 0:
                    delta = -shares
                elif not CFG["fractional_shares"]:
                    delta = float(math.trunc(delta))
                if delta > 0:
                    px = o * (1 + slip); delta = min(delta, cash / (px * (1 + comm)))
                    if not CFG["fractional_shares"]: delta = float(math.floor(delta))
                elif delta < 0:
                    px = o * (1 - slip)
                else:
                    px = o
                if abs(delta) * px > 1e-9:
                    notional = abs(delta) * px; fee = notional * comm
                    if old_u == 0:
                        cyc_cf, cyc_cost, cyc_start = 0.0, 0.0, dates[i]
                    if delta > 0:
                        cash -= notional + fee; cyc_cf -= notional + fee; cyc_cost += notional + fee
                    else:
                        cash += notional - fee; cyc_cf += notional - fee
                    shares += delta
                    trk.apply_fill(eff, px)
                    fills.append(dict(date=dates[i], side=ACTIONS[eff], shares=delta, price=px, notional=notional, fee=fee,
                                      slippage_cost=abs(delta) * abs(px - o), units_before=old_u, units_after=trk.units))
                    if trk.units == 0:
                        shares = 0.0
                        cycles.append(dict(start=cyc_start, end=dates[i], pnl=cyc_cf, ret=cyc_cf / max(cyc_cost, 1e-12)))
        equity = cash + shares * c
        trk.mark(c); pos_hist[i] = trk.features(c, S["atr"][i])
        action = decider(i, trk, pos_hist) if i < n - 1 else HOLD
        pending = action
        rec.append(dict(date=dates[i], equity=equity, pos_value=shares * c, units=trk.units, action=int(action),
                        regime_prev=int(S["regime"][i - 1]), regime_now=int(S["regime"][i]), close=c))
    daily = pd.DataFrame(rec).set_index("date")
    daily["ret"] = daily["equity"].pct_change()
    return dict(daily=daily, fills=pd.DataFrame(fills), cycles=pd.DataFrame(cycles), open_units_end=trk.units)


# ---------- deciders ----------
def model_decider(policy, pack):
    X, L_ = pack["X"], policy.seq_len
    def f(i, trk, pos_hist):
        if i < L_ - 1:
            return HOLD
        w = X[i - L_ + 1: i + 1].copy()
        w[:, POS_SLICE] = pos_hist[i - L_ + 1: i + 1]       # live position state replaces the teacher's
        return policy.predict(w, normalized=True, mask_invalid=True)["predicted_action"]
    return f

def rule_decider(pack):
    rule = TrendRule(CFG)
    return lambda i, trk, ph: rule.decide(pack["S"], i, pack["S"]["regime"][i], trk)

def sma_cross_decider(pack):
    S = pack["S"]
    def f(i, trk, ph):
        on = S["sma_med"][i] > S["sma_long"][i]
        return BUY if (on and trk.units == 0) else EXIT if (not on and trk.units > 0) else HOLD
    return f

def buy_hold_decider(pack):
    return lambda i, trk, ph: BUY if trk.units == 0 else HOLD

POLICY = TrendFollowingPolicy(MODEL, make_meta(MODEL_ARGS, SCALER), DEVICE)

## 13. Run the backtests (held-out test period only)
The policy is evaluated bar by bar (its own position feeds back into its inputs), so this section takes a few minutes on CPU/GPU, more with cost sensitivity enabled.

In [ ]:
def run_all(name, decider_fn, buy_step, cost_mult=1.0):
    t0 = time.time()
    res = {t: run_backtest(PACKS[t], decider_fn(PACKS[t]), buy_step, cost_mult) for t in ASSETS}
    print(f"  {name:34s} done in {time.time() - t0:5.1f}s")
    return res

print("Running test-period backtests ...")
RESULTS = {
    "SP-03 policy":         run_all("SP-03 policy", lambda pk: model_decider(POLICY, pk), 1),
    "Rule teacher (proxy)": run_all("Rule teacher (proxy)", rule_decider, 1),
    "SMA50/200 baseline":   run_all("SMA50/200 baseline", sma_cross_decider, CFG["max_units"]),
    "Buy & hold (EW)":      run_all("Buy & hold (EW)", buy_hold_decider, CFG["max_units"]),
}
BENCH_RES = run_backtest(BENCH_PACK, buy_hold_decider(BENCH_PACK), CFG["max_units"], 1.0, cap=CFG["initial_capital"])
STRATS = list(RESULTS.keys()) + [f"Benchmark B&H ({CFG['benchmark']})"]

def aggregate(res):
    eq = pd.concat({t: r["daily"]["equity"] for t, r in res.items()}, axis=1).sort_index().ffill()
    eq = eq.fillna(CFG["initial_capital"] / len(res))
    pv = pd.concat({t: r["daily"]["pos_value"] for t, r in res.items()}, axis=1).sort_index().ffill().fillna(0.0)
    inm = pd.concat({t: (r["daily"]["units"] > 0).astype(float) for t, r in res.items()}, axis=1).sort_index().ffill().fillna(0.0)
    total = eq.sum(axis=1)
    fills = pd.concat([r["fills"].assign(ticker=t) for t, r in res.items() if len(r["fills"])], ignore_index=True) if any(len(r["fills"]) for r in res.values()) else pd.DataFrame()
    cycles = pd.concat([r["cycles"].assign(ticker=t) for t, r in res.items() if len(r["cycles"])], ignore_index=True) if any(len(r["cycles"]) for r in res.values()) else pd.DataFrame()
    return dict(total=total, per_ticker=eq, exposure=pv.sum(axis=1) / total, time_in_market=inm.mean(axis=1), fills=fills, cycles=cycles)

def perf_metrics(agg, initial_capital):
    eq = agg["total"]; r = eq.pct_change().dropna()
    years = max((eq.index[-1] - eq.index[0]).days / 365.25, 1e-9)
    tot = eq.iloc[-1] / initial_capital - 1
    sd = r.std(); rf_d = CFG["rf_annual"] / CFG["trading_days"]
    dn = np.minimum(r - rf_d, 0); dd_std = np.sqrt((dn ** 2).mean())
    dd = eq / eq.cummax() - 1
    f, cy = agg["fills"], agg["cycles"]
    out = dict(
        total_return=float(tot), cagr=float((1 + tot) ** (1 / years) - 1) if tot > -1 else -1.0,
        ann_vol=float(sd * np.sqrt(CFG["trading_days"])),
        sharpe=float((r.mean() - rf_d) / sd * np.sqrt(CFG["trading_days"])) if sd > 0 else float("nan"),
        sortino=float((r.mean() - rf_d) / dd_std * np.sqrt(CFG["trading_days"])) if dd_std > 0 else float("nan"),
        max_drawdown=float(dd.min()), calmar=float(((1 + tot) ** (1 / years) - 1) / abs(dd.min())) if dd.min() < 0 else float("nan"),
        worst_21d_return=float((eq / eq.shift(21) - 1).min()),
        avg_exposure=float(agg["exposure"].mean()), time_in_market=float(agg["time_in_market"].mean()),
        n_fills=int(len(f)), n_round_trips=int(len(cy)),
        win_rate_round_trips=float((cy["pnl"] > 0).mean()) if len(cy) else float("nan"),
        turnover_annual=float(f["notional"].sum() / eq.mean() / years) if len(f) else 0.0,
        total_costs_pct_of_capital=float((f["fee"].sum() + f["slippage_cost"].sum()) / initial_capital) if len(f) else 0.0,
        start=str(eq.index[0].date()), end=str(eq.index[-1].date()), years=float(years),
    )
    return out

AGG = {k: aggregate(v) for k, v in RESULTS.items()}
bench_total = BENCH_RES["daily"]["equity"]
AGG[STRATS[-1]] = dict(total=bench_total, per_ticker=bench_total.to_frame(CFG["benchmark"]), exposure=BENCH_RES["daily"]["pos_value"] / bench_total,
                       time_in_market=(BENCH_RES["daily"]["units"] > 0).astype(float),
                       fills=BENCH_RES["fills"], cycles=BENCH_RES["cycles"])
METRICS = {k: perf_metrics(a, CFG["initial_capital"]) for k, a in AGG.items()}
MET_DF = pd.DataFrame(METRICS).T
show_cols = ["total_return", "cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "calmar", "worst_21d_return", "avg_exposure",
             "time_in_market", "n_fills", "n_round_trips", "win_rate_round_trips", "turnover_annual", "total_costs_pct_of_capital"]
print(f"\nTEST-PERIOD RESULTS ({METRICS['SP-03 policy']['start']} -> {METRICS['SP-03 policy']['end']}), "
      f"costs: {CFG['commission_bps']} bps commission + {CFG['slippage_bps']} bps slippage per side")
print(MET_DF[show_cols].astype(float).round(4).to_string())

In [ ]:
# Cost sensitivity for the policy (robustness to execution assumptions)
COST_SENS = {}
if CFG["run_cost_sensitivity"]:
    print("Cost sensitivity (policy only) ...")
    for m in CFG["cost_multipliers"]:
        if m == 1.0:
            COST_SENS[m] = METRICS["SP-03 policy"]; continue
        res = run_all(f"policy @ cost x{m}", lambda pk: model_decider(POLICY, pk), 1, cost_mult=m)
        COST_SENS[m] = perf_metrics(aggregate(res), CFG["initial_capital"])
    print(pd.DataFrame(COST_SENS).T[["total_return", "sharpe", "max_drawdown", "turnover_annual", "total_costs_pct_of_capital"]].astype(float).round(4))

## 14. Performance across regimes, years, tickers — including the failures
Regime attribution uses the regime known at the **previous** close (the one the position was decided under), so it is causal.

In [ ]:
def regime_perf(res_dict):
    rows = []
    for name, res in res_dict.items():
        d = pd.concat([r["daily"].dropna(subset=["ret"]).assign(ticker=t) for t, r in res.items()])
        d["expo"] = d["pos_value"] / d["equity"]
        for rid, rname in REGIMES.items():
            s = d[d.regime_prev == rid]
            if len(s) < 5:
                rows.append(dict(strategy=name, regime=rname, bars=len(s))); continue
            sd = s["ret"].std()
            rows.append(dict(strategy=name, regime=rname, bars=len(s), mean_daily_bps=1e4 * s["ret"].mean(),
                             ann_return_arith=s["ret"].mean() * CFG["trading_days"], sharpe_like=(s["ret"].mean() / sd * np.sqrt(CFG["trading_days"])) if sd > 0 else np.nan,
                             hit_rate=(s["ret"] > 0).mean(), avg_exposure=s["expo"].mean()))
    return pd.DataFrame(rows)

REGIME_PERF = regime_perf(RESULTS)
print("Pooled (all assets) daily performance by regime in force at the previous close:")
print(REGIME_PERF.round(4).to_string(index=False))

# yearly returns
def yearly(eq, start_cap):
    last = eq.groupby(eq.index.year).last()
    prev = pd.concat([pd.Series([start_cap]), last.iloc[:-1]]).values
    return pd.Series(last.values / prev - 1, index=last.index)

YEARLY = pd.DataFrame({k: yearly(a["total"], CFG["initial_capital"]) for k, a in AGG.items()})
YEARLY["policy_minus_EW_B&H"] = YEARLY["SP-03 policy"] - YEARLY["Buy & hold (EW)"]
print("\nCalendar-year returns (partial first/last years are labelled by calendar year only):")
print((YEARLY * 100).round(2).to_string())

# drawdown episodes
def drawdown_episodes(eq, top=3):
    dd = eq / eq.cummax() - 1; is_dd = dd < 0; grp = (is_dd != is_dd.shift()).cumsum()
    eps = []
    for _, seg in dd[is_dd].groupby(grp[is_dd]):
        s_pos = eq.index.get_loc(seg.index[0]); e_pos = eq.index.get_loc(seg.index[-1])
        eps.append(dict(peak_date=eq.index[max(s_pos - 1, 0)].date(), trough_date=seg.idxmin().date(), depth=float(seg.min()),
                        last_underwater_date=seg.index[-1].date(), recovered=bool(e_pos + 1 < len(eq)), bars_underwater=int(len(seg))))
    return pd.DataFrame(eps).sort_values("depth").head(top) if eps else pd.DataFrame()

print("\nWorst drawdown episodes — SP-03 policy:"); DD_POLICY = drawdown_episodes(AGG["SP-03 policy"]["total"]); print(DD_POLICY.to_string(index=False))
print("\nWorst drawdown episodes — Buy & hold (EW):"); print(drawdown_episodes(AGG["Buy & hold (EW)"]["total"]).to_string(index=False))

# per-ticker
rows = []
for t in ASSETS:
    row = dict(ticker=t)
    for name in ["SP-03 policy", "SMA50/200 baseline", "Buy & hold (EW)"]:
        e = AGG[name]["per_ticker"][t]; r = e.pct_change().dropna()
        row[f"{name} ret"] = e.iloc[-1] / e.iloc[0] - 1
        row[f"{name} maxDD"] = (e / e.cummax() - 1).min()
    row["policy_trips"] = len(RESULTS["SP-03 policy"][t]["cycles"]); row["policy_ret - B&H ret"] = row["SP-03 policy ret"] - row["Buy & hold (EW) ret"]
    rows.append(row)
PER_TICKER = pd.DataFrame(rows).set_index("ticker")
print("\nPer-ticker (test period):"); print(PER_TICKER.round(4).to_string())
print("\nTickers where the policy LOST money:", PER_TICKER.index[PER_TICKER["SP-03 policy ret"] < 0].tolist() or "none")
print("Tickers where the policy underperformed buy & hold:", PER_TICKER.index[PER_TICKER["policy_ret - B&H ret"] < 0].tolist() or "none")

In [ ]:
# Plots
colors = {"SP-03 policy": "C0", "Buy & hold (EW)": "C1", "SMA50/200 baseline": "C2", "Rule teacher (proxy)": "C3", STRATS[-1]: "gray"}
fig, ax = plt.subplots(3, 1, figsize=(13, 13), gridspec_kw={"height_ratios": [3, 1.5, 1.2]}, sharex=True)
for k, a in AGG.items():
    eq = a["total"] / CFG["initial_capital"]; ax[0].plot(eq.index, eq, label=k, color=colors[k], lw=2 if k == "SP-03 policy" else 1.2, ls="--" if "Benchmark" in k else "-")
    ax[1].plot(eq.index, eq / eq.cummax() - 1, color=colors[k], lw=1)
ax[0].set_title("Held-out test: growth of 1.0 (after costs & slippage)"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title("Drawdown"); ax[1].grid(alpha=.3)
ax[2].plot(AGG["SP-03 policy"]["exposure"], label="policy exposure"); ax[2].plot(AGG["SMA50/200 baseline"]["exposure"], label="SMA baseline", alpha=.6)
ax[2].set_title("Exposure (position value / equity)"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

# regime-shaded example
ex = ASSETS[0]; pk = PACKS[ex]; d = RESULTS["SP-03 policy"][ex]["daily"]; S_ = pk["S"]
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(d.index, d["close"], color="k", lw=1, label=f"{ex} close")
sm = pd.Series(S_["sma_long"], index=pk["dates"]).reindex(d.index); sm2 = pd.Series(S_["sma_med"], index=pk["dates"]).reindex(d.index)
ax[0].plot(sm.index, sm, label="SMA200", color="purple", lw=1); ax[0].plot(sm2.index, sm2, label="SMA50", color="orange", lw=1)
reg_colors = {TRENDING: "green", WEAKENING: "gold", SIDEWAYS: "lightgray"}
for rid, col in reg_colors.items():
    m = (d["regime_now"] == rid).values
    ax[0].fill_between(d.index, d["close"].min(), d["close"].max(), where=m, color=col, alpha=.2, label=f"regime: {REGIMES[rid]}")
ax[0].legend(ncol=3, fontsize=8); ax[0].set_title(f"{ex}: price, MAs and regimes (test)")
ax[1].step(d.index, d["units"], where="post"); ax[1].set_title("SP-03 policy units held"); ax[1].set_yticks(range(CFG["max_units"] + 1))
plt.tight_layout(); plt.show()

In [ ]:
# Auto-generated, non-promotional summary — numbers only, flags computed from the results above.
m, bh, bl, rt = METRICS["SP-03 policy"], METRICS["Buy & hold (EW)"], METRICS["SMA50/200 baseline"], METRICS["Rule teacher (proxy)"]
print("SUMMARY (test period; research only, not a forecast)\n" + "-" * 60)
print(f"Policy total return {m['total_return']:+.1%}, Sharpe {m['sharpe']:.2f}, max drawdown {m['max_drawdown']:.1%}, turnover {m['turnover_annual']:.1f}x/yr, {m['n_round_trips']} round trips.")
print(f"EW buy&hold         {bh['total_return']:+.1%}, Sharpe {bh['sharpe']:.2f}, max drawdown {bh['max_drawdown']:.1%}.")
print(f"SMA50/200 baseline  {bl['total_return']:+.1%}, Sharpe {bl['sharpe']:.2f}, max drawdown {bl['max_drawdown']:.1%}.")
print(f"Rule teacher        {rt['total_return']:+.1%}, Sharpe {rt['sharpe']:.2f}, max drawdown {rt['max_drawdown']:.1%}.")
flags = []
if m["total_return"] < bh["total_return"]: flags.append("policy return BELOW equal-weight buy & hold")
if m["sharpe"] < bh["sharpe"]: flags.append("policy Sharpe BELOW buy & hold")
if m["total_return"] < bl["total_return"]: flags.append("policy return BELOW the simple SMA baseline")
if m["max_drawdown"] < bh["max_drawdown"]: flags.append("policy max drawdown WORSE than buy & hold")
if m["total_return"] < rt["total_return"]: flags.append("policy return BELOW the rule it imitates (imitation loss)")
if m["total_return"] < 0: flags.append("policy LOST money over the test period")
if (YEARLY["SP-03 policy"] < 0).any(): flags.append(f"losing calendar years: {YEARLY.index[YEARLY['SP-03 policy'] < 0].tolist()}")
if COST_SENS and 4.0 in COST_SENS and COST_SENS[4.0]["total_return"] < 0: flags.append("return turns negative at 4x costs")
print("Flags:", flags if flags else "none triggered")
print("Interpretation guardrails: one test window, finite sample, survivorship-biased ticker list, adjusted-price data, costs are assumptions.")

## 15. Export the standardized artifacts
Files written to `sp03_artifacts/`: `trend_following_policy.pt`, `trend_following_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json`, `equity_curve.csv`.

In [ ]:
from datetime import date

def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, float): return None if not np.isfinite(o) else o
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (pd.Timestamp, datetime, date)): return str(o)
    if isinstance(o, pd.DataFrame): return to_py(o.reset_index().to_dict(orient="records"))
    if isinstance(o, pd.Series): return to_py(o.to_dict())
    return o

# 1) model artifact
POLICY.save(OUT / "trend_following_policy.pt")

# 2) config
config_out = to_py({
    "config": CFG, "model_args": MODEL_ARGS, "env": ENV_INFO, "created_utc": datetime.now(timezone.utc).isoformat(),
    "assets_used": ASSETS, "download_failures": DOWNLOAD_REPORT[DOWNLOAD_REPORT.status != "OK"][["symbol", "status", "error"]],
    "label_mode": LABEL_MODE, "label_note": "PROXY labels from a causal trend-following rule unless human_actions_csv was supplied.",
    "split": {"train_end": CFG["train_end"], "val_end": CFG["val_end"], "test_start": str(TEST_START.date())},
    "training": {"best_epoch": best_epoch, "epochs_run": len(HIST["epoch"]), "best_val_loss": best_val, "seconds": TRAIN_SECONDS,
                 "class_weights": {ACTIONS[i]: float(CLASS_W[i]) for i in range(4)}},
    "schema_hash": SCHEMA_HASH,
    "data_manifest": DOWNLOAD_REPORT[DOWNLOAD_REPORT.status == "OK"][["symbol", "rows", "first", "last", "fingerprint"]],
    "teacher_rule_params": {k: CFG[k] for k in ["adx_trend", "adx_add", "recent_trend_window", "atr_stop_mult", "weak_exit_bars", "cooldown_bars", "min_hold_add", "max_units"]},
    "disclaimer": "Research and paper-trading only. No guarantee of profitability.",
})
(OUT / "trend_following_config.json").write_text(json.dumps(config_out, indent=2))

# 3) feature schema
schema = {
    "policy_id": CFG["policy_id"], "schema_version": 1, "schema_hash": SCHEMA_HASH, "seq_len": CFG["seq_len"],
    "state_shape": [CFG["seq_len"], len(FEATURE_COLUMNS)], "column_order": FEATURE_COLUMNS,
    "columns": ([dict(index=i, name=n_, group=g, description=d_, normalized=True, source="market (causal)") for i, (n_, g, d_) in enumerate(FEATURE_REGISTRY)] +
                [dict(index=N_MARKET + i, name=n_, group=g, description=d_, normalized=False, source="position state supplied by the trading environment") for i, (n_, g, d_) in enumerate(POSITION_REGISTRY)] +
                [dict(index=N_MARKET + N_POS + i, name=c, group="trader", description="static trader/risk-profile feature in [0,1]", normalized=False, source="user profile") for i, c in enumerate(TRADER_COLUMNS)]),
    "normalization": {"method": "StandardScaler fitted on training period only, then clipped", "clip": CFG["feature_clip"], "applies_to": MARKET_COLUMNS},
    "action_map": {str(k): v for k, v in ACTIONS.items()},
    "action_semantics": {"HOLD": "keep position", "BUY": "add one unit (open if flat); invalid at max units",
                         "SELL": "trim ONE unit (partial reduction); invalid when flat", "EXIT": "close the entire position; invalid when flat"},
    "position_model": {"direction": "long-only", "max_units": CFG["max_units"], "unit_fraction_of_position_limit": 1.0 / CFG["max_units"]},
    "regime_map": {str(k): v for k, v in REGIMES.items()},
    "timing": "features at close of bar t -> action executed at open of bar t+1",
    "causality": "all market features are backward-looking; verified by a truncation test in the notebook",
    "label_source": LABEL_MODE,
}
(OUT / "feature_schema.json").write_text(json.dumps(to_py(schema), indent=2))

# 4) preprocessing (plain dict + sklearn scaler; the .pt file also embeds the statistics so it is self-contained)
with open(OUT / "preprocessing.pkl", "wb") as fh:
    pickle.dump({"scaler": SCALER, "market_columns": MARKET_COLUMNS, "passthrough_columns": POSITION_COLUMNS + TRADER_COLUMNS,
                 "feature_columns": FEATURE_COLUMNS, "clip": CFG["feature_clip"], "seq_len": CFG["seq_len"], "schema_hash": SCHEMA_HASH,
                 "fit_period_end": CFG["train_end"], "sklearn_version": sklearn.__version__}, fh)

# 5) evaluation metrics
eval_out = to_py({
    "label_mode": LABEL_MODE,
    "note": "Classification metrics measure agreement with the labels (proxy rule unless human actions supplied). Backtest metrics are the economic evidence.",
    "classification": {sp: {k: EVAL[sp][k] for k in ["acc", "bal_acc", "macro_f1", "majority_acc", "invalid_rate", "report", "cm"]} for sp in EVAL},
    "regime_classification": REGIME_EVAL, "confidence_calibration_test": CAL.reset_index(drop=True),
    "backtest": {"period": {"start": METRICS["SP-03 policy"]["start"], "end": METRICS["SP-03 policy"]["end"]},
                 "assumptions": {k: CFG[k] for k in ["initial_capital", "commission_bps", "slippage_bps", "max_position_fraction", "fractional_shares", "rf_annual"]},
                 "metrics": METRICS, "cost_sensitivity": COST_SENS, "regime_performance": REGIME_PERF, "yearly_returns": YEARLY,
                 "per_ticker": PER_TICKER, "worst_drawdown_episodes_policy": DD_POLICY, "flags": flags},
    "download_failures": DOWNLOAD_REPORT[DOWNLOAD_REPORT.status != "OK"][["symbol", "status", "error"]],
    "limitations": ["single chronological test window", "proxy labels imitate a rule", "survivorship-biased ticker list", "yfinance adjusted data",
                    "assumed costs/slippage", "long-only, no borrowing/shorting", "not investment advice"],
})
(OUT / "evaluation_metrics.json").write_text(json.dumps(eval_out, indent=2))

# 6) equity curve
ec = pd.DataFrame({"date": AGG["SP-03 policy"]["total"].index})
ec = ec.set_index("date")
for k, key in [("SP-03 policy", "strategy_equity"), ("Buy & hold (EW)", "buy_hold_ew_equity"), ("SMA50/200 baseline", "sma_baseline_equity"),
               ("Rule teacher (proxy)", "rule_teacher_equity"), (STRATS[-1], "benchmark_equity")]:
    ec[key] = AGG[k]["total"].reindex(ec.index).ffill()
ec["strategy_drawdown"] = ec["strategy_equity"] / ec["strategy_equity"].cummax() - 1
ec["strategy_exposure"] = AGG["SP-03 policy"]["exposure"].reindex(ec.index)
for t in ASSETS:
    ec[f"eq_{t}"] = AGG["SP-03 policy"]["per_ticker"][t].reindex(ec.index).ffill()
ec.to_csv(OUT / "equity_curve.csv")

for f_ in sorted(OUT.iterdir()):
    print(f"{f_.name:32s} {f_.stat().st_size / 1024:9.1f} KB")

## 16. Verify the artifact: reload and predict with no data download, no retraining

In [ ]:
loaded = TrendFollowingPolicy.load(OUT / "trend_following_policy.pt", DEVICE)
assert loaded.feature_columns == FEATURE_COLUMNS and loaded.meta["schema_hash"] == SCHEMA_HASH

rng = np.random.default_rng(0)
check_idx = rng.choice(IDX["test"], size=min(20, len(IDX["test"])), replace=False)
max_dp = 0.0
for e in check_idx:
    # raw (un-normalized) state in schema order, exactly what an external caller would hold
    raw = np.hstack([X_MARKET_RAW[e - L + 1: e + 1], X_POS[e - L + 1: e + 1], X_TRADER[e - L + 1: e + 1]])
    raw_df = pd.DataFrame(raw, columns=FEATURE_COLUMNS)
    a = loaded.predict(raw_df)                                  # normalization applied internally
    b = POLICY.predict(X_ALL[e - L + 1: e + 1], normalized=True)  # in-memory model, normalized path
    max_dp = max(max_dp, float(np.abs(np.array(a["action_probabilities"]) - np.array(b["action_probabilities"])).max()))
    assert abs(sum(a["action_probabilities"]) - 1) < 1e-5 and 0 <= a["confidence"] <= 1 and a["predicted_action"] in ACTIONS
print(f"Reloaded-vs-in-memory max |Δprob| over {len(check_idx)} windows: {max_dp:.2e}")
assert max_dp < 1e-4

ex_out = loaded.predict(pd.DataFrame(np.hstack([X_MARKET_RAW[check_idx[0] - L + 1: check_idx[0] + 1], X_POS[check_idx[0] - L + 1: check_idx[0] + 1], X_TRADER[check_idx[0] - L + 1: check_idx[0] + 1]]), columns=FEATURE_COLUMNS))
print(json.dumps(ex_out, indent=2))

# end-to-end: backtest one asset with the reloaded policy and confirm identical results
t_chk = ASSETS[0]
r_re = run_backtest(PACKS[t_chk], model_decider(loaded, PACKS[t_chk]), 1)
eq_a, eq_b = RESULTS["SP-03 policy"][t_chk]["daily"]["equity"], r_re["daily"]["equity"]
print(f"Reloaded-policy backtest vs original ({t_chk}): max relative equity difference = {np.abs(eq_a / eq_b - 1).max():.2e}")

In [ ]:
# Package everything for download
archive = shutil.make_archive("sp03_trend_following_artifacts", "zip", OUT)
print("Created", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("Not running in Colab (or download blocked); find the zip in the working directory.", repr(exc)[:80])

## 17. Integration checklist for the Meta-RL selector

**Contract**
- [ ] Action IDs are `0=HOLD, 1=BUY, 2=SELL, 3=EXIT`; SP-03 `SELL` = trim one unit, `EXIT` = close all. Confirm the other specialized policies and the Meta-RL environment use the same meanings (especially if another policy treats SELL as a full exit).
- [ ] `predict()` returns `action_probabilities` (len 4, sums to 1), `predicted_action`, `confidence`. Confirm the selector reads these keys (and decide whether to use `confidence` or `max_probability`; `confidence` is a learned reliability head trained on proxy-label agreement, **not** a calibrated probability of profit).
- [ ] Compare `schema_hash` and `column_order` in `feature_schema.json` with the other policies. If their schemas differ, the selector must build **per-policy state tensors** (shared columns are not guaranteed identical in definition — e.g. `ret_5`, `rsi_14`, `macd*` scalings).
- [ ] State shape is `(seq_len=60, n_features)`; the selector must provide the **last 60 bars** per asset in schema order, with market columns **un-normalized** (or pass `normalized=True` after applying `preprocessing.pkl` / embedded statistics).

**Position-aware inputs (critical)**
- [ ] The environment must supply the four position columns for every bar in the window (`position_fraction`, `unrealized_pnl`, `holding_duration`, `stop_distance_atr`), computed with the same definitions as `PositionTracker`. Wrong or zeroed position inputs silently degrade the policy.
- [ ] Use `valid_action_mask` / `mask_invalid=True` (or equivalent) so BUY at max exposure and SELL/EXIT while flat become HOLD.
- [ ] Map units → size consistently: `max_units=2`, 1 unit = 50 % of the per-asset position limit.

**Timing & execution**
- [ ] Decisions are made at the close of bar *t* and must be executed at the **next open** (or later). Do not feed the policy bar *t+1* data when scoring bar *t*.
- [ ] Apply realistic costs/slippage in the Meta-RL reward; SP-03 turnover and costs are reported in `evaluation_metrics.json`.

**Data & labels**
- [ ] Remember labels are **proxy (rule-derived)** unless `label_mode` says human actions were used. The policy is a smooth, learned version of a trend rule, not an independent alpha source.
- [ ] Re-check the regime-specific results: expect weaker behaviour in `SIDEWAYS` (whipsaw) and during sharp reversals. The Meta-RL selector should be allowed to down-weight SP-03 in those regimes.
- [ ] Optional trader/risk-profile columns are static and only informative if the policy is retrained with real, varying profiles.

**Governance**
- [ ] Verify `trend_following_policy.pt` loads with `TrendFollowingPolicy.load(path, device)` on the target machine (CPU and GPU) and reproduces outputs (cell in section 16).
- [ ] Retrain / re-evaluate periodically; log data fingerprints (`data_manifest` in the config JSON).
- [ ] Paper trading only until independent out-of-sample and live-paper evidence exists. Nothing in this notebook is investment advice or a promise of profit.